# 03 — Logistic Regression From Scratch vs Sklearn
Implementasi LR from scratch, visualisasi loss contour & parameter trajectory, perbandingan optimizer.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt

from src.utils import set_seed, save_model
from src.data import load_train
from src.cleaning import DataCleaner
from src.preprocessing import Preprocessor
from src.algorithms.logistic_regression import LogisticRegressionScratch
from src.optimizers import GradientDescent, Adam
from src.evaluation import cross_validate, macro_f1_score, stratified_k_fold_indices
from src.sklearn_baselines import get_sklearn_lr
from src.visualization import plot_loss_curves, plot_loss_contour_and_trajectory
from src import config

set_seed(42)

In [ ]:
# Load & preprocess
train_df = load_train()
cleaner = DataCleaner()
train_clean = cleaner.fit_transform(train_df)
preprocessor = Preprocessor()
X_train, y_train = preprocessor.fit_transform(train_clean)
print(f'X_train: {X_train.shape}')

In [ ]:
# Compare GD vs Adam convergence
folds = stratified_k_fold_indices(y_train, n_folds=5)
tr_idx, val_idx = folds[0]
X_tr, y_tr = X_train[tr_idx], y_train[tr_idx]
X_val, y_val = X_train[val_idx], y_train[val_idx]

# Train with GD
lr_gd = LogisticRegressionScratch(
    max_iter=500, batch_size=512, lambda_reg=0.01,
    class_weight='balanced', optimizer=GradientDescent(lr=0.01)
)
lr_gd.fit(X_tr, y_tr)

# Train with Adam
lr_adam = LogisticRegressionScratch(
    max_iter=500, batch_size=512, lambda_reg=0.01,
    class_weight='balanced', optimizer=Adam(lr=0.01)
)
lr_adam.fit(X_tr, y_tr)

print(f'GD - Final loss: {lr_gd.loss_history[-1]:.4f}, Macro F1: {macro_f1_score(y_val, lr_gd.predict(X_val)):.4f}')
print(f'Adam - Final loss: {lr_adam.loss_history[-1]:.4f}, Macro F1: {macro_f1_score(y_val, lr_adam.predict(X_val)):.4f}')

In [ ]:
# Plot loss convergence comparison
import os
plot_loss_curves(
    {'Gradient Descent': lr_gd.loss_history, 'Adam': lr_adam.loss_history},
    title='Logistic Regression: Loss Convergence — GD vs Adam',
    save_path=os.path.join(config.FIGURES_DIR, 'lr_convergence_gd_vs_adam.png')
)

In [ ]:
# Loss contour & parameter trajectory (projected to 2D)
# Using dimensions for previous_loan_defaults_on_file_Yes and loan_percent_income
feat_names = preprocessor.feature_names_
dim1_name = 'previous_loan_defaults_on_file_Yes'
dim2_name = 'loan_percent_income'
dim1 = feat_names.index(dim1_name) if dim1_name in feat_names else 0
dim2 = feat_names.index(dim2_name) if dim2_name in feat_names else 1

plot_loss_contour_and_trajectory(
    X_tr, y_tr, lr_adam.w_history,
    dim1=dim1, dim2=dim2,
    feature_names=feat_names,
    title='LR Loss Contour + Adam Trajectory',
    save_path=os.path.join(config.FIGURES_DIR, 'lr_loss_contour_trajectory.png'),
    lambda_reg=0.01
)

In [ ]:
# Cross-validate LR from scratch (with threshold tuning)
def lr_factory():
    return LogisticRegressionScratch(
        max_iter=500, batch_size=512, lambda_reg=0.01,
        class_weight='balanced', optimizer=Adam(lr=0.01)
    )

lr_cv = cross_validate(lr_factory, X_train, y_train, n_folds=5, tune_threshold=True)
print(f'LR From-Scratch (Adam + threshold tuning):')
print(f'  Mean Macro F1: {lr_cv["mean_f1"]:.4f} +/- {lr_cv["std_f1"]:.4f}')

In [ ]:
# Sklearn baseline
lr_sk_cv = cross_validate(get_sklearn_lr, X_train, y_train, n_folds=5)
print(f'Sklearn LogisticRegression:')
print(f'  Mean Macro F1: {lr_sk_cv["mean_f1"]:.4f} +/- {lr_sk_cv["std_f1"]:.4f}')

In [ ]:
# Save best model
lr_final = lr_factory()
lr_final.fit(X_train, y_train)
save_model(lr_final, 'logistic_regression_adam.pkl')
print('Model saved.')